In [2]:
!pip install -q ultralytics open-clip-torch hnswlib tqdm pandas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00


In [3]:
# ============================================================
#  NOTEBOOK A — Condition A: Vision-Only CLIP (Baseline)
#  alpha = 1  →  pure image embedding, no captions, no fine-tuning
#
#  Seeds  : 039, 003, 113, 528  (team roll-numbers)
#
#  Pipeline:
#    1. Load dataset splits (gallery / query)
#    2. Load YOLO (your fine-tuned weights — do NOT re-train)
#    3. Load CLIP (pretrained)
#    4. YOLO-crop every gallery image (with padding)
#    5. Encode crops with CLIP vision encoder
#    6. Build HNSW index (hnswlib, cosine space)
#    7. Evaluate: HR@K, Recall@K, NDCG@K, mAP@K  (K ∈ {5, 10, 15})
#       — repeated over 4 seeds, query order shuffled per seed
#    8. Aggregate metrics across seeds
# ============================================================

# 

# ── Cell 1: Imports ──────────────────────────────────────────
import os, json, random, time
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
import pandas as pd
import hnswlib
import open_clip
from ultralytics import YOLO
from torchvision import transforms

NOTEBOOK_START = time.time()

# ── Cell 2: Config ───────────────────────────────────────────
CFG = dict(
    # ---- dataset paths (same layout as offline_pipeline_final.py) ----
    img_root   = "/kaggle/input/datasets/fireworksbads/project2/project2/img/img",
    eval_file  = "/kaggle/input/datasets/fireworksbads/project2/project2/eval/list_eval_partition.txt",
    bbox_file  = "/kaggle/input/datasets/fireworksbads/project2/project2/anno/list_bbox_inshop.txt",

    # ---- YOUR fine-tuned YOLO (already trained — read-only) ----------
    yolo_weights = "/kaggle/input/datasets/fireworksbads/output/detect/train/weights/best.pt",

    # ---- CLIP settings -----------------------------------------------
    clip_model    = "ViT-B-32",
    clip_pretrain = "openai",
    image_size    = 224,

    # ---- working dirs ------------------------------------------------
    crop_dir  = "/kaggle/working/crops_A",
    index_dir = "/kaggle/working/indexes_A",

    # ---- HNSW (hnswlib) ----------------------------------------------
    hnsw_M           = 32,
    hnsw_efConstruct = 200,
    hnsw_efSearch    = 50,

    # ---- misc --------------------------------------------------------
    batch_size = 32,
    yolo_pad   = 10,
    top_k      = [5, 10, 15],
    seeds      = [39, 3, 113, 528],   # team roll-numbers (same as B & C)
    device     = "cuda" if torch.cuda.is_available() else "cpu",
)

os.makedirs(CFG["crop_dir"],  exist_ok=True)
os.makedirs(CFG["index_dir"], exist_ok=True)
print("Device:", CFG["device"])


def _elapsed_h():
    return (time.time() - NOTEBOOK_START) / 3600


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Device: cuda


In [ ]:

# ── Cell 3: Read eval partition (same logic as offline_pipeline) ─
split_df = pd.read_csv(
    CFG["eval_file"],
    sep=r"\s+",
    header=0,
    names=["image_name", "item_id", "evaluation_status"],
    skiprows=1,
)
split_df = split_df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

gallery_df = split_df[split_df["evaluation_status"] == "gallery"].reset_index(drop=True)
query_df   = split_df[split_df["evaluation_status"] == "query"].reset_index(drop=True)

gallery_paths  = [os.path.join(CFG["img_root"], p) for p in gallery_df["image_name"]]
gallery_items  = gallery_df["item_id"].tolist()
gallery_item_counts = {}
for it in gallery_items:
    gallery_item_counts[it] = gallery_item_counts.get(it, 0) + 1


query_paths    = [os.path.join(CFG["img_root"], p) for p in query_df["image_name"]]
query_items    = query_df["item_id"].tolist()

print(f"Gallery: {len(gallery_df)}   Query: {len(query_df)}")

# ── Cell 4: Load YOLO ────────────────────────────────────────
model_yolo = YOLO(CFG["yolo_weights"])
model_yolo.to(CFG["device"])
print("YOLO loaded.")


def yolo_crop_batch(img_paths, save_dir, pad=None):
    """Batch YOLO crop with padding; selects largest-area box; fallback to full image."""
    if pad is None:
        pad = CFG["yolo_pad"]
    os.makedirs(save_dir, exist_ok=True)
    results   = model_yolo.predict(img_paths, device=CFG["device"], verbose=False)
    out_paths = []
    for img_path, result in zip(img_paths, results):
        img   = Image.open(img_path).convert("RGB")
        fname = img_path.replace("/", "_").replace("\\", "_").lstrip("_") + ".jpg"
        out   = os.path.join(save_dir, fname)
        boxes = result.boxes
        if boxes is not None and len(boxes) > 0:
            # Select largest box by area (matching pipeline_part1.ipynb)
            areas = (boxes.xyxy[:, 2] - boxes.xyxy[:, 0]) * (boxes.xyxy[:, 3] - boxes.xyxy[:, 1])
            best = boxes[areas.argmax()]
            x1, y1, x2, y2 = map(int, best.xyxy[0].tolist())
            W, H = img.size
            img.crop((max(0, x1 - pad), max(0, y1 - pad),
                      min(W, x2 + pad), min(H, y2 + pad))).save(out)
        else:
            img.save(out)
        out_paths.append(out)
    return out_paths


In [ ]:

# ── Cell 5: Load CLIP (pretrained) ───────────────────────────
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    CFG["clip_model"], pretrained=CFG["clip_pretrain"]
)
clip_model = clip_model.to(CFG["device"]).eval()
print("CLIP (pretrained) loaded.")

eval_tf = transforms.Compose([
    transforms.Resize((CFG["image_size"], CFG["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073),
                         std=(0.26862954, 0.26130258, 0.27577711)),
])


@torch.no_grad()
def get_image_emb(model, img_path):
    """Return L2-normalized CLIP image embedding (float32 numpy)."""
    img    = Image.open(img_path).convert("RGB")
    tensor = eval_tf(img).unsqueeze(0).to(CFG["device"])
    with torch.amp.autocast(CFG["device"], enabled=(CFG["device"] != "cpu")):
        feat = model.encode_image(tensor)
    return F.normalize(feat[0], dim=-1).cpu().numpy().astype("float32")

# ── Cell 6: YOLO-crop gallery images ─────────────────────────
print("\nCropping gallery images with YOLO...")
gallery_crops = []
BS = CFG["batch_size"]

for i in tqdm(range(0, len(gallery_paths), BS), desc="YOLO crop gallery"):
    batch = gallery_paths[i:i + BS]
    try:
        crops = yolo_crop_batch(batch, CFG["crop_dir"])
        gallery_crops.extend(crops)
    except Exception as e:
        print(f"  [skip batch {i}]: {e}")
        gallery_crops.extend(batch)   # fallback: use original

print(f"Gallery crops ready: {len(gallery_crops)}")

# ── Cell 7: Encode gallery with CLIP ─────────────────────────
print("\nEncoding gallery images with CLIP (Condition A — alpha=1)...")
gallery_vecs = []
gallery_meta = []   # stores {item_id, image_path, cropped_path}

for img_path, crop_path, item_id in tqdm(
        zip(gallery_paths, gallery_crops, gallery_items), total=len(gallery_paths),
        desc="CLIP encode gallery"):
    try:
        emb = get_image_emb(clip_model, crop_path)
        gallery_vecs.append(emb)
        gallery_meta.append({
            "item_id":      item_id,
            "image_path":   img_path,
            "cropped_path": crop_path,
        })
    except Exception as e:
        print(f"  [skip] {img_path}: {e}")


In [ ]:

gallery_vecs_np = np.array(gallery_vecs, dtype="float32")
print(f"Gallery embeddings shape: {gallery_vecs_np.shape}")

# ── Cell 8: Build HNSW index (hnswlib, cosine space) ─────────
print("\nBuilding HNSW index (hnswlib, cosine space)...")
dim = gallery_vecs_np.shape[1]
index_A = hnswlib.Index(space='cosine', dim=dim)
index_A.init_index(max_elements=len(gallery_vecs_np),
                   ef_construction=CFG["hnsw_efConstruct"],
                   M=CFG["hnsw_M"])
index_A.add_items(gallery_vecs_np, np.arange(len(gallery_vecs_np)))
index_A.set_ef(CFG["hnsw_efSearch"])

index_path = os.path.join(CFG["index_dir"], "index_A.bin")
meta_path  = os.path.join(CFG["index_dir"], "meta_A.json")
index_A.save_index(index_path)
with open(meta_path, "w") as f:
    json.dump(gallery_meta, f, indent=2)

print(f"Index A saved → {index_path}  ({len(gallery_vecs_np)} vectors)")

# ── Cell 9: YOLO-crop query images (once) ────────────────────
print("\nCropping query images with YOLO...")
query_crops = []
for i in tqdm(range(0, len(query_paths), BS), desc="YOLO crop queries"):
    batch = query_paths[i:i + BS]
    try:
        crops = yolo_crop_batch(batch, CFG["crop_dir"])
        query_crops.extend(crops)
    except Exception as e:
        query_crops.extend(batch)


In [ ]:

# ── Cell 10: Encode query images with CLIP (once) ─────────────
print("\nEncoding query images with CLIP...")
query_vecs = []
for qpath, qcrop in tqdm(zip(query_paths, query_crops), total=len(query_paths),
                          desc="CLIP encode queries"):
    try:
        query_vecs.append(get_image_emb(clip_model, qcrop))
    except Exception as e:
        print(f"  [skip] {qpath}: {e}")
        query_vecs.append(np.zeros(gallery_vecs_np.shape[1], dtype="float32"))

query_vecs_np = np.array(query_vecs, dtype="float32")
print(f"Query embeddings shape: {query_vecs_np.shape}")

# ── Cell 11: Metric helpers (TWO recall formulas) ─────────────
def hit_rate_at_k(retrieved, rel_set, k):
    """Hit Rate@K (binary): 1 if any relevant item appears in top-K, else 0."""
    return int(len(set(retrieved[:k]) & rel_set) > 0)


def recall_at_k(retrieved, rel_set, k, nr):
    """Recall@K (proportion): # relevant items in top-K / total relevant in gallery."""
    hits = sum(1 for r in retrieved[:k] if r in rel_set)
    return hits / max(1, nr)


def ap_at_k(retrieved, rel_set, k, nr):
    """Average Precision@K."""
    hits, cum = 0, 0.0
    for rank, r in enumerate(retrieved[:k], 1):
        if r in rel_set:
            hits += 1
            cum  += hits / rank
    return cum / max(1, min(k, nr))


def ndcg_at_k(retrieved, rel_set, k, nr):
    """NDCG@K."""
    dcg  = sum(1.0 / np.log2(i + 2) for i, r in enumerate(retrieved[:k]) if r in rel_set)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(k, nr)))
    return dcg / idcg if idcg > 0 else 0.0


def compute_metrics(all_ret, all_rel, all_nr, top_k):
    """Compute HR@K, Recall@K, mAP@K, NDCG@K for all K values."""
    out = {}
    for k in top_k:
        out[k] = {
            "HR":     (float(np.mean([hit_rate_at_k(r, rel, k) for r, rel in zip(all_ret, all_rel)])),
                       float(np.std ([hit_rate_at_k(r, rel, k) for r, rel in zip(all_ret, all_rel)]))),
            "Recall": (float(np.mean([recall_at_k(r, rel, k, nr) for r, rel, nr in zip(all_ret, all_rel, all_nr)])),
                       float(np.std ([recall_at_k(r, rel, k, nr) for r, rel, nr in zip(all_ret, all_rel, all_nr)]))),
            "mAP":    (float(np.mean([ap_at_k(r, rel, k, nr) for r, rel, nr in zip(all_ret, all_rel, all_nr)])),
                       float(np.std ([ap_at_k(r, rel, k, nr) for r, rel, nr in zip(all_ret, all_rel, all_nr)]))),
            "NDCG":   (float(np.mean([ndcg_at_k(r, rel, k, nr) for r, rel, nr in zip(all_ret, all_rel, all_nr)])),
                       float(np.std ([ndcg_at_k(r, rel, k, nr) for r, rel, nr in zip(all_ret, all_rel, all_nr)]))),
        }
    return out


In [ ]:

# ── Cell 12: Main loop — 4 seeds ──────────────────────────────
# The HNSW index and gallery embeddings are deterministic (no randomness).
# Seeds shuffle the query order → gives variance estimates over ordering.

all_results_A = {}
max_k = max(CFG["top_k"])

for seed in CFG["seeds"]:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    print(f"\n{'='*62}")
    print(f"  SEED {seed:03d}   [{_elapsed_h():.2f}h elapsed]")
    print(f"{'='*62}")

    perm       = np.random.permutation(len(query_paths))
    q_paths_s  = [query_paths[i]  for i in perm]
    q_items_s  = [query_items[i]  for i in perm]
    q_embs_s   = query_vecs_np[perm]

    run_key = f"seed{seed:03d}"

    # Retrieve using hnswlib
    all_ret, all_rel = [], []
    for qi in tqdm(range(len(q_paths_s)), desc=f"  Search seed={seed}", leave=False):
        try:
            qvec = q_embs_s[qi].reshape(1, -1).astype("float32")
            labels, distances = index_A.knn_query(qvec, k=max_k)
            retrieved_items = [gallery_meta[i]["item_id"] for i in labels[0]]
            all_ret.append(retrieved_items)
            all_rel.append({q_items_s[qi]})   # set-based relevance
        except Exception as e:
            print(f"  [skip query] {q_paths_s[qi]}: {e}")

    all_nr = [gallery_item_counts.get(list(rel)[0], 1) for rel in all_rel]
    metrics = compute_metrics(all_ret, all_rel, all_nr, CFG["top_k"])
    all_results_A[run_key] = metrics
    for k in CFG["top_k"]:
        hr, hrs = metrics[k]["HR"]
        r, rs   = metrics[k]["Recall"]
        m, ms   = metrics[k]["mAP"]
        n, ns   = metrics[k]["NDCG"]
        print(f"    K={k:2d}  HR={hr:.4f}±{hrs:.4f}  R={r:.4f}±{rs:.4f}  mAP={m:.4f}±{ms:.4f}  NDCG={n:.4f}±{ns:.4f}")

# ── Cell 13: Aggregate across seeds ───────────────────────────
print(f"\n\n{'='*80}")
print("  CONDITION A — Aggregated over 4 seeds")
print(f"{'='*80}")
print(f"{'Config':<28} {'K':>3}  {'HR@K':>12}  {'Recall@K':>12}  {'mAP':>12}  {'NDCG':>12}")
print("-" * 84)

sms = [all_results_A[f"seed{s:03d}"] for s in CFG["seeds"]
       if f"seed{s:03d}" in all_results_A]

aggregated_A = {}
if sms:
    for k in CFG["top_k"]:
        aggregated_A[k] = {
            met: (float(np.mean([sm[k][met][0] for sm in sms])),
                  float(np.std( [sm[k][met][0] for sm in sms])))
            for met in ["HR", "Recall", "mAP", "NDCG"]
        }
        hr, hrs = aggregated_A[k]["HR"]
        r, rs   = aggregated_A[k]["Recall"]
        m, ms   = aggregated_A[k]["mAP"]
        n, ns   = aggregated_A[k]["NDCG"]
        print(f"  A (vision-only)          {k:>3}  {hr:.4f}±{hrs:.4f}  {r:.4f}±{rs:.4f}  {m:.4f}±{ms:.4f}  {n:.4f}±{ns:.4f}")

# ── Cell 14: Save results ─────────────────────────────────────
results_path = os.path.join(CFG["index_dir"], "results_A.json")
with open(results_path, "w") as f:
    json.dump({
        "seeds":      CFG["seeds"],
        "per_seed":   {k: {str(kk): {m: list(v) for m, v in vv.items()}
                           for kk, vv in vs.items()}
                       for k, vs in all_results_A.items()},
        "aggregated": {str(k): {m: list(v) for m, v in vv.items()}
                       for k, vv in aggregated_A.items()},
    }, f, indent=2)
print(f"\nResults saved → {results_path}")
print("✅ Notebook A complete.")